# Analysis Without Nemotron

This notebook answers one robustness question:

> Are the main findings still visible after removing `nemotron-3-nano-4b`?

The notebook reads the raw JSONL result files, joins prompt metadata, filters out Nemotron, and recomputes core summaries without relying on the precomputed pooled CSVs.

In [1]:
from __future__ import annotations

import csv
import html
import json
import math
import re
from collections import defaultdict
from pathlib import Path
from statistics import mean, median
from IPython.display import HTML, Markdown, display

ROOT = Path('/Users/macos/LearningSpace/ue-courses/Thesis/02ScalingPhrase/localmodel')
RESULTS_DIR = ROOT / 'results'
PROBLEMS_DIR = ROOT / 'problems'
ANALYSIS_DIR = RESULTS_DIR / 'analysis'
EXCLUDED_MODEL = 'nemotron-3-nano-4b_1_50'

def fnum(value, default=0.0):
    try:
        if value in ('', None):
            return default
        return float(value)
    except (TypeError, ValueError):
        return default

def pct(value):
    return f'{fnum(value) * 100:.1f}%'

def pp(value):
    return f'{fnum(value):+.1f} pp'

def model_slug(path: Path) -> str:
    return path.name.replace('.math.jsonl', '').replace('.jsonl', '')

def variant_from_task_id(task_id: str) -> int:
    match = re.search(r'_(\d+)$', str(task_id or ''))
    return int(match.group(1)) if match else 0

def base_from_task_id(task_id: str) -> str:
    return re.sub(r'_\d+$', '', str(task_id or ''))

def normalize_role(role: str) -> str:
    return 'role' if role and role.strip().lower() != 'no role' else 'no role'

def safe_mean(values):
    return mean(values) if values else 0.0

def safe_median(values):
    return median(values) if values else 0.0

def two_prop_p(success_a, total_a, success_b, total_b):
    if total_a == 0 or total_b == 0:
        return 1.0
    pooled = (success_a + success_b) / (total_a + total_b)
    variance = pooled * (1 - pooled) * (1 / total_a + 1 / total_b)
    if variance <= 0:
        return 1.0
    z = abs((success_a / total_a - success_b / total_b) / math.sqrt(variance))
    return math.erfc(z / math.sqrt(2))

def grouped(rows, keys):
    groups = defaultdict(list)
    for row in rows:
        groups[tuple(row[k] for k in keys)].append(row)
    return groups

def table(rows, columns, formats=None, max_rows=20):
    formats = formats or {}
    visible = rows[:max_rows]
    head = ''.join(f'<th>{html.escape(c)}</th>' for c in columns)
    body = []
    for row in visible:
        cells = []
        for col in columns:
            value = row.get(col, '')
            if formats.get(col) == 'pct':
                value = pct(value)
            elif formats.get(col) == 'pp':
                value = pp(value)
            elif isinstance(value, float):
                value = f'{value:.3f}'
            cells.append(f'<td>{html.escape(str(value))}</td>')
        body.append(f'<tr>{"".join(cells)}</tr>')
    note = f'<p><em>Showing {len(visible)} of {len(rows)} rows.</em></p>' if len(rows) > len(visible) else ''
    css = '<style>table.insight{border-collapse:collapse;font-size:13px}.insight th,.insight td{border:1px solid #ddd;padding:5px 7px;text-align:left}.insight th{background:#f4f4f4}.insight tr:nth-child(even){background:#fafafa}</style>'
    display(HTML(css + f'<table class="insight"><thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table>{note}'))

def shorten(text, max_chars=42):
    text = str(text)
    return text if len(text) <= max_chars else text[:max_chars-1] + '…'

BAR_CHART_COUNTER = 0

def bar_chart(rows, label_key, value_key, title, value_format='pct', width=1040, height=None, color='#2f6f73', left=360, right=140):
    global BAR_CHART_COUNTER
    BAR_CHART_COUNTER += 1
    clip_id = f'labelClip{BAR_CHART_COUNTER}'
    rows = list(rows)
    if not rows:
        display(Markdown('_No rows to plot._'))
        return
    height = height or max(280, 58 + len(rows) * 34)
    top, bottom = 64, 32
    inner_w = width - left - right
    bar_h = max(15, (height - top - bottom) / len(rows) - 8)
    values = [fnum(r[value_key]) for r in rows]
    labels = [pct(v) if value_format == 'pct' else pp(v) if value_format == 'pp' else f'{v:.2f}' for v in values]
    min_v, max_v = min(values), max(values)
    span = max(max_v - min_v, 0.001)
    zero_x = left if min_v >= 0 else left + inner_w * ((0 - min_v) / span)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="#fbfaf7"/>', f'<text x="{left}" y="34" font-family="Arial" font-size="20" font-weight="700">{html.escape(title)}</text>', f'<clipPath id="{clip_id}"><rect x="0" y="{top-12}" width="{left-18}" height="{height-top-bottom+24}"/></clipPath>']
    if min_v < 0 < max_v:
        parts.append(f'<line x1="{zero_x:.1f}" x2="{zero_x:.1f}" y1="{top-8}" y2="{height-bottom}" stroke="#555" stroke-width="1"/>')
    for i, row in enumerate(rows):
        y = top + i * (bar_h + 8)
        value = values[i]
        if min_v < 0:
            x0 = left + inner_w * ((min(value, 0) - min_v) / span)
            x1 = left + inner_w * ((max(value, 0) - min_v) / span)
            bar_w = max(1, x1 - x0)
        else:
            x0 = left
            bar_w = max(1, inner_w * (value / max(max_v, 0.001)))
        label = html.escape(shorten(row[label_key]))
        full = html.escape(str(row[label_key]))
        parts.append(f'<text clip-path="url(#{clip_id})" x="{left-28}" y="{y + bar_h*.68:.1f}" text-anchor="end" font-family="Arial" font-size="12"><title>{full}</title>{label}</text>')
        parts.append(f'<rect x="{x0:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" rx="3" fill="{color}"/>')
        if value < 0:
            tx = x0 + 10
            anchor = 'start'
            text_fill = '#ffffff' if bar_w >= 55 else '#111111'
        else:
            tx = min(width - 8, x0 + bar_w + 10)
            anchor = 'start'
            text_fill = '#111111'
        parts.append(f'<text x="{tx:.1f}" y="{y + bar_h*.68:.1f}" text-anchor="{anchor}" font-family="Arial" font-size="12" fill="{text_fill}">{labels[i]}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))

def load_csv(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

MODEL_SIZES = load_csv(ANALYSIS_DIR / 'model_size_comparison.csv')


In [2]:
def load_template_metadata():
    metadata = {}
    for kind, path in [('code', PROBLEMS_DIR / 'prompt_matrix_prompt_template.09.csv'), ('math', PROBLEMS_DIR / 'prompt_matrix_prompt_template.09.math.csv')]:
        with path.open('r', encoding='utf-8', newline='') as handle:
            for row in csv.DictReader(handle):
                role = row.get('Role framing', '')
                metadata[(kind, int(row['ID']))] = {
                    'tone': row.get('Tone', '').strip(),
                    'role_binary': normalize_role(role),
                    'role_framing': role.strip(),
                    'format': row.get('Format', '').strip(),
                }
    return metadata

def read_rows(include_nemotron=True):
    meta = load_template_metadata()
    rows = []
    for path in sorted(RESULTS_DIR.glob('*.jsonl')):
        kind = 'math' if path.name.endswith('.math.jsonl') else 'code'
        slug = model_slug(path)
        if not include_nemotron and slug == EXCLUDED_MODEL:
            continue
        with path.open('r', encoding='utf-8') as handle:
            for line in handle:
                if not line.strip():
                    continue
                row = json.loads(line)
                variant_id = int(row.get('category_id') or variant_from_task_id(row.get('task_id')))
                if kind == 'code':
                    total = int(row.get('tests_total') or 0)
                    passed_units = int(row.get('tests_passed') or 0)
                else:
                    total = int(row.get('values_total') or 0)
                    passed_units = int(row.get('values_passed') or 0)
                out = {
                    'kind': kind,
                    'model_slug': slug,
                    'variant_id': variant_id,
                    'base_id': row.get('base_task_id') or base_from_task_id(row.get('task_id')),
                    'passed': bool(row.get('passed')),
                    'score': passed_units / total if total else float(bool(row.get('passed'))),
                    'prompt_tokens': fnum(row.get('prompt_tokens')),
                    'completion_tokens': fnum(row.get('completion_tokens')),
                    'duration_seconds': fnum(row.get('duration_seconds')),
                }
                out.update(meta.get((kind, variant_id), {}))
                rows.append(out)
    return rows

all_rows = read_rows(include_nemotron=True)
rows = read_rows(include_nemotron=False)
display(Markdown(f'Loaded `{len(all_rows)}` total rows and `{len(rows)}` rows after excluding `{EXCLUDED_MODEL}`.'))

Loaded `10800` total rows and `9000` rows after excluding `nemotron-3-nano-4b_1_50`.

## 0. Model Size Comparison

All size values below are taken from local Ollama manifests. Nemotron is included here as context even though the analysis below excludes it.

In [3]:
if MODEL_SIZES:
    size_rows = sorted(MODEL_SIZES, key=lambda r: fnum(r['nominal_parameters_b']))
    table(
        size_rows,
        ['model_tag', 'nominal_parameters_b', 'local_or_registry_gib', 'size_source', 'notes'],
        max_rows=10,
    )
    for row in size_rows:
        row['label'] = row['model_tag']
    bar_chart(size_rows, 'label', 'local_or_registry_gib', 'Local Ollama model size in GiB', value_format='number', height=380, color='#7c6fbd')
else:
    display(Markdown('_No model_size_comparison.csv found._'))


model_tag,nominal_parameters_b,local_or_registry_gib,size_source,notes
nemotron-3-nano:4b,3.97,2.643,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/nemotron-3-nano/4b.
deepseek-coder:6.7b,6.74,3.565,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/deepseek-coder/6.7b.
qwen2.5:7b,7,4.361,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/qwen2.5/7b.
mistral:7b-instruct,7.24,4.073,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/mistral/7b-instruct.
llama3.1:8b,8,4.583,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/llama3.1/8b.
gemma2:9b,9,5.069,local Ollama manifest,Measured from local manifest model layer size: registry.ollama.ai/library/gemma2/9b.


In [4]:
def summarize(rows, keys):
    out = []
    for key_values, items in sorted(grouped(rows, keys).items()):
        record = {k: v for k, v in zip(keys, key_values)}
        record.update({
            'n': len(items),
            'passed': sum(1 for r in items if r['passed']),
            'pass_rate': safe_mean([1.0 if r['passed'] else 0.0 for r in items]),
            'mean_score': safe_mean([r['score'] for r in items]),
            'mean_prompt_tokens': safe_mean([r['prompt_tokens'] for r in items]),
            'mean_completion_tokens': safe_mean([r['completion_tokens'] for r in items]),
            'mean_duration_seconds': safe_mean([r['duration_seconds'] for r in items]),
        })
        out.append(record)
    return out

def factor_contrasts(rows):
    baselines = {'tone': 'neutral', 'role_binary': 'no role', 'format': 'plaintext'}
    out = []
    for group_keys in [('kind',), ('model_slug', 'kind')]:
        for factor, baseline in baselines.items():
            summaries = summarize(rows, (*group_keys, factor))
            base_by_group = {tuple(r[k] for k in group_keys): r for r in summaries if r.get(factor) == baseline}
            for r in summaries:
                group = tuple(r[k] for k in group_keys)
                base = base_by_group.get(group)
                if not base or r.get(factor) == baseline:
                    continue
                rec = {k: r[k] for k in group_keys}
                rec.update({
                    'factor': factor,
                    'baseline': baseline,
                    'level': r[factor],
                    'baseline_pass_rate': base['pass_rate'],
                    'level_pass_rate': r['pass_rate'],
                    'delta_pass_rate_pp': (r['pass_rate'] - base['pass_rate']) * 100,
                    'z_test_p_value': two_prop_p(r['passed'], r['n'], base['passed'], base['n']),
                    'delta_score': r['mean_score'] - base['mean_score'],
                    'delta_prompt_tokens': r['mean_prompt_tokens'] - base['mean_prompt_tokens'],
                    'delta_completion_tokens': r['mean_completion_tokens'] - base['mean_completion_tokens'],
                    'delta_duration_seconds': r['mean_duration_seconds'] - base['mean_duration_seconds'],
                })
                out.append(rec)
    return out

def kind_differences(contrasts):
    model_rows = [r for r in contrasts if 'model_slug' in r]
    by_key = {(r['model_slug'], r['factor'], r['level'], r['kind']): r for r in model_rows}
    out = []
    for model, factor, level in sorted(set((r['model_slug'], r['factor'], r['level']) for r in model_rows)):
        code = by_key.get((model, factor, level, 'code'))
        math = by_key.get((model, factor, level, 'math'))
        if not code or not math:
            continue
        out.append({
            'model_slug': model,
            'factor': factor,
            'level': level,
            'code_delta_pp': code['delta_pass_rate_pp'],
            'math_delta_pp': math['delta_pass_rate_pp'],
            'math_minus_code_delta_pp': math['delta_pass_rate_pp'] - code['delta_pass_rate_pp'],
            'math_minus_code_prompt_tokens': math['delta_prompt_tokens'] - code['delta_prompt_tokens'],
            'math_minus_code_completion_tokens': math['delta_completion_tokens'] - code['delta_completion_tokens'],
            'math_minus_code_duration_seconds': math['delta_duration_seconds'] - code['delta_duration_seconds'],
            'code_p_value': code['z_test_p_value'],
            'math_p_value': math['z_test_p_value'],
        })
    return sorted(out, key=lambda r: -abs(r['math_minus_code_delta_pp']))

by_kind = summarize(rows, ('kind',))
by_model_kind = summarize(rows, ('model_slug', 'kind'))
contrasts = factor_contrasts(rows)
kind_diff = kind_differences(contrasts)

## 0. Model Comparison Without Nemotron

This section appears before factor analysis to show how the model landscape changes after removing the dominant Nemotron model.

In [5]:
model_compare = sorted(summarize(rows, ('kind', 'model_slug')), key=lambda r: (r['kind'], -r['pass_rate']))
for row in model_compare:
    row['model_short'] = row['model_slug'].replace('_1_50', '')
    row['label'] = f"{row['kind']} / {row['model_short']}"

bar_chart(model_compare, 'label', 'pass_rate', 'Model performance by task kind without Nemotron', height=520, color='#3f7fbf')
table(
    model_compare,
    ['kind', 'model_short', 'n', 'passed', 'pass_rate', 'mean_score', 'mean_prompt_tokens', 'mean_completion_tokens', 'mean_duration_seconds'],
    {'pass_rate': 'pct'},
    max_rows=12,
)

full_compare = []
full_by_model_kind = summarize(all_rows, ('kind', 'model_slug'))
without_lookup = {(r['kind'], r['model_slug']): r for r in model_compare}
for full in full_by_model_kind:
    if full['model_slug'] == EXCLUDED_MODEL:
        status = 'excluded'
    else:
        status = 'kept'
    full_compare.append({
        'status': status,
        'kind': full['kind'],
        'model_short': full['model_slug'].replace('_1_50', ''),
        'full_pass_rate': full['pass_rate'],
        'without_nemotron_pass_rate': without_lookup.get((full['kind'], full['model_slug']), {}).get('pass_rate', ''),
        'full_mean_score': full['mean_score'],
        'full_mean_completion_tokens': full['mean_completion_tokens'],
        'full_mean_duration_seconds': full['mean_duration_seconds'],
    })
full_compare = sorted(full_compare, key=lambda r: (r['kind'], r['status'], -fnum(r['full_pass_rate'])))
table(full_compare, ['status', 'kind', 'model_short', 'full_pass_rate', 'without_nemotron_pass_rate', 'full_mean_score', 'full_mean_completion_tokens', 'full_mean_duration_seconds'], {'full_pass_rate': 'pct', 'without_nemotron_pass_rate': 'pct'}, max_rows=14)


kind,model_short,n,passed,pass_rate,mean_score,mean_prompt_tokens,mean_completion_tokens,mean_duration_seconds
code,qwen2.5-7b,900,119,13.2%,0.173,282.540,315.996,9.198
code,gemma2-9b,900,102,11.3%,0.194,289.347,175.218,6.230
code,llama3.1-8b,900,52,5.8%,0.109,268.660,393.932,11.640
code,deepseek-coder-6.7b-instruct,900,10,1.1%,0.020,313.031,491.087,9.725
code,mistral-7b-instruct,900,4,0.4%,0.025,301.911,366.227,9.586
math,llama3.1-8b,900,389,43.2%,0.432,157.493,273.569,8.437
math,qwen2.5-7b,900,212,23.6%,0.236,158.293,5.556,0.730
math,gemma2-9b,900,137,15.2%,0.152,159.860,6.101,0.826
math,mistral-7b-instruct,900,69,7.7%,0.077,167.398,78.651,2.326
math,deepseek-coder-6.7b-instruct,900,51,5.7%,0.057,174.184,287.059,5.885


status,kind,model_short,full_pass_rate,without_nemotron_pass_rate,full_mean_score,full_mean_completion_tokens,full_mean_duration_seconds
excluded,code,nemotron-3-nano-4b,22.8%,0.0%,0.304,1646.110,27.080
kept,code,qwen2.5-7b,13.2%,13.2%,0.173,315.996,9.198
kept,code,gemma2-9b,11.3%,11.3%,0.194,175.218,6.230
kept,code,llama3.1-8b,5.8%,5.8%,0.109,393.932,11.640
kept,code,deepseek-coder-6.7b-instruct,1.1%,1.1%,0.020,491.087,9.725
kept,code,mistral-7b-instruct,0.4%,0.4%,0.025,366.227,9.586
excluded,math,nemotron-3-nano-4b,91.2%,0.0%,0.912,288.629,5.120
kept,math,llama3.1-8b,43.2%,43.2%,0.432,273.569,8.437
kept,math,qwen2.5-7b,23.6%,23.6%,0.236,5.556,0.730
kept,math,gemma2-9b,15.2%,15.2%,0.152,6.101,0.826


## 1. Overall Result Without Nemotron

In [6]:
table(by_kind, ['kind', 'n', 'passed', 'pass_rate', 'mean_score', 'mean_prompt_tokens', 'mean_completion_tokens', 'mean_duration_seconds'], {'pass_rate': 'pct'})
bar_chart(by_kind, 'kind', 'pass_rate', 'Pass rate by task kind without Nemotron')

kind,n,passed,pass_rate,mean_score,mean_prompt_tokens,mean_completion_tokens,mean_duration_seconds
code,4500,287,6.4%,0.104,291.098,348.492,9.276
math,4500,858,19.1%,0.191,163.446,130.187,3.641


## 2. Model Ranking Without Nemotron

In [7]:
mk = sorted(by_model_kind, key=lambda r: (r['kind'], -r['pass_rate']))
for r in mk:
    r['label'] = f"{r['kind']} / {r['model_slug'].replace('_1_50', '')}"
bar_chart(mk, 'label', 'pass_rate', 'Pass rate by model and kind without Nemotron', height=500, color='#3f7fbf')
table(mk, ['kind', 'model_slug', 'n', 'passed', 'pass_rate', 'mean_score', 'mean_prompt_tokens', 'mean_completion_tokens', 'mean_duration_seconds'], {'pass_rate': 'pct'}, max_rows=12)

kind,model_slug,n,passed,pass_rate,mean_score,mean_prompt_tokens,mean_completion_tokens,mean_duration_seconds
code,qwen2.5-7b_1_50,900,119,13.2%,0.173,282.540,315.996,9.198
code,gemma2-9b_1_50,900,102,11.3%,0.194,289.347,175.218,6.230
code,llama3.1-8b_1_50,900,52,5.8%,0.109,268.660,393.932,11.640
code,deepseek-coder-6.7b-instruct_1_50,900,10,1.1%,0.020,313.031,491.087,9.725
code,mistral-7b-instruct_1_50,900,4,0.4%,0.025,301.911,366.227,9.586
math,llama3.1-8b_1_50,900,389,43.2%,0.432,157.493,273.569,8.437
math,qwen2.5-7b_1_50,900,212,23.6%,0.236,158.293,5.556,0.730
math,gemma2-9b_1_50,900,137,15.2%,0.152,159.860,6.101,0.826
math,mistral-7b-instruct_1_50,900,69,7.7%,0.077,167.398,78.651,2.326
math,deepseek-coder-6.7b-instruct_1_50,900,51,5.7%,0.057,174.184,287.059,5.885


## 3. Factor Effects Without Nemotron

These are raw contrasts versus baseline: neutral tone, no role, plaintext format.

In [8]:
overall = [r for r in contrasts if 'model_slug' not in r]
overall = sorted(overall, key=lambda r: (r['kind'], r['factor'], -abs(r['delta_pass_rate_pp'])))
for r in overall:
    r['label'] = f"{r['kind']} / {r['factor']}={r['level']}"
bar_chart(overall, 'label', 'delta_pass_rate_pp', 'Factor pass-rate deltas without Nemotron', value_format='pp', height=390, color='#7c6fbd')
table(overall, ['kind', 'factor', 'baseline', 'level', 'baseline_pass_rate', 'level_pass_rate', 'delta_pass_rate_pp', 'delta_prompt_tokens', 'delta_completion_tokens', 'delta_duration_seconds', 'z_test_p_value'], {'baseline_pass_rate': 'pct', 'level_pass_rate': 'pct', 'delta_pass_rate_pp': 'pp'}, max_rows=12)

kind,factor,baseline,level,baseline_pass_rate,level_pass_rate,delta_pass_rate_pp,delta_prompt_tokens,delta_completion_tokens,delta_duration_seconds,z_test_p_value
code,format,plaintext,xml,6.3%,6.5%,+0.3 pp,26.920,-12.477,-0.282,0.765
code,format,plaintext,json,6.3%,6.3%,+0.1 pp,17.041,-13.912,-0.326,0.940
code,role_binary,no role,role,6.8%,5.9%,-0.9 pp,3.400,8.783,0.219,0.200
code,tone,neutral,polite,6.6%,6.0%,-0.6 pp,29.933,49.465,1.314,0.499
code,tone,neutral,aggressive,6.6%,6.5%,-0.1 pp,65.000,-1.895,0.008,0.941
math,format,plaintext,json,21.7%,16.4%,-5.3 pp,13.977,-36.221,-1.077,0.000
math,format,plaintext,xml,21.7%,19.1%,-2.5 pp,21.832,1.839,-0.027,0.085
math,role_binary,no role,role,17.9%,20.3%,+2.4 pp,3.600,12.050,0.336,0.040
math,tone,neutral,polite,17.9%,20.2%,+2.3 pp,31.733,20.306,0.661,0.104
math,tone,neutral,aggressive,17.9%,19.1%,+1.3 pp,64.200,8.781,0.344,0.372


## 4. Math Versus Code Differences Without Nemotron

This repeats the important math-vs-code interaction view, but removes the strongest math model.

In [9]:
kd = sorted(kind_diff, key=lambda r: -abs(r['math_minus_code_delta_pp']))
for r in kd:
    r['label'] = f"{r['model_slug'].replace('_1_50', '')} / {r['factor']}={r['level']}"
bar_chart(kd[:15], 'label', 'math_minus_code_delta_pp', 'Math-vs-code pass-rate difference without Nemotron', value_format='pp', height=560, color='#2f6f73')
table(kd, ['model_slug', 'factor', 'level', 'code_delta_pp', 'math_delta_pp', 'math_minus_code_delta_pp', 'math_minus_code_prompt_tokens', 'math_minus_code_completion_tokens', 'math_minus_code_duration_seconds', 'code_p_value', 'math_p_value'], {'code_delta_pp': 'pp', 'math_delta_pp': 'pp', 'math_minus_code_delta_pp': 'pp'}, max_rows=20)

for metric, title, color in [
    ('math_minus_code_prompt_tokens', 'Math-vs-code prompt-token effect without Nemotron', '#7c6fbd'),
    ('math_minus_code_completion_tokens', 'Math-vs-code completion-token effect without Nemotron', '#c75d4c'),
    ('math_minus_code_duration_seconds', 'Math-vs-code duration effect without Nemotron', '#d89b34'),
]:
    metric_rows = sorted(kind_diff, key=lambda r: -abs(r[metric]))
    for r in metric_rows:
        r['label'] = f"{r['model_slug'].replace('_1_50', '')} / {r['factor']}={r['level']}"
    bar_chart(metric_rows[:15], 'label', metric, title, value_format='number', height=560, color=color)

model_slug,factor,level,code_delta_pp,math_delta_pp,math_minus_code_delta_pp,math_minus_code_prompt_tokens,math_minus_code_completion_tokens,math_minus_code_duration_seconds,code_p_value,math_p_value
llama3.1-8b_1_50,format,json,-0.3 pp,-15.7 pp,-15.3 pp,-3.380,-110.463,-3.188,0.858,0.000
qwen2.5-7b_1_50,role_binary,role,-1.1 pp,+10.2 pp,+11.3 pp,-1.000,-21.258,-0.580,0.623,0.000
llama3.1-8b_1_50,format,xml,+0.7 pp,-9.7 pp,-10.3 pp,-5.040,28.723,0.819,0.731,0.018
llama3.1-8b_1_50,tone,polite,-2.7 pp,+5.3 pp,+8.0 pp,3.000,-14.933,-0.390,0.169,0.188
deepseek-coder-6.7b-instruct_1_50,format,xml,+0.3 pp,-5.7 pp,-6.0 pp,-5.140,4.123,-0.517,0.653,0.004
qwen2.5-7b_1_50,format,xml,+0.0 pp,+5.7 pp,+5.7 pp,-5.040,-13.303,-0.352,1.000,0.108
deepseek-coder-6.7b-instruct_1_50,format,json,+1.0 pp,-4.3 pp,-5.3 pp,-2.640,1.113,-0.548,0.254,0.035
deepseek-coder-6.7b-instruct_1_50,tone,polite,+0.3 pp,+4.7 pp,+4.3 pp,1.000,11.890,0.781,0.563,0.013
gemma2-9b_1_50,format,json,-1.0 pp,-5.3 pp,-4.3 pp,-3.440,3.747,0.148,0.697,0.072
gemma2-9b_1_50,tone,aggressive,-2.0 pp,+2.3 pp,+4.3 pp,-1.000,-7.433,-0.137,0.434,0.426


## 5. Compare With Full Dataset

This final table compares full-dataset task-kind summaries to summaries after removing Nemotron.

In [10]:
full_kind = summarize(all_rows, ('kind',))
compare = []
without_by_kind = {r['kind']: r for r in by_kind}
for full in full_kind:
    w = without_by_kind[full['kind']]
    compare.append({
        'kind': full['kind'],
        'full_pass_rate': full['pass_rate'],
        'without_nemotron_pass_rate': w['pass_rate'],
        'delta_without_minus_full_pp': (w['pass_rate'] - full['pass_rate']) * 100,
        'full_mean_score': full['mean_score'],
        'without_nemotron_mean_score': w['mean_score'],
        'full_mean_completion_tokens': full['mean_completion_tokens'],
        'without_nemotron_completion_tokens': w['mean_completion_tokens'],
        'full_mean_duration_seconds': full['mean_duration_seconds'],
        'without_nemotron_duration_seconds': w['mean_duration_seconds'],
    })
table(compare, ['kind', 'full_pass_rate', 'without_nemotron_pass_rate', 'delta_without_minus_full_pp', 'full_mean_score', 'without_nemotron_mean_score', 'full_mean_completion_tokens', 'without_nemotron_completion_tokens', 'full_mean_duration_seconds', 'without_nemotron_duration_seconds'], {'full_pass_rate': 'pct', 'without_nemotron_pass_rate': 'pct', 'delta_without_minus_full_pp': 'pp'})

display(Markdown('### Interpretation'))
display(Markdown('- If math performance drops sharply without Nemotron, then Nemotron was driving the pooled math advantage.\n- If factor directions remain similar, then the finding is robust.\n- If factor directions flip, then the previous conclusion should be written as model-dependent rather than general.'))

kind,full_pass_rate,without_nemotron_pass_rate,delta_without_minus_full_pp,full_mean_score,without_nemotron_mean_score,full_mean_completion_tokens,without_nemotron_completion_tokens,full_mean_duration_seconds,without_nemotron_duration_seconds
code,9.1%,6.4%,-2.7 pp,0.137,0.104,564.761,348.492,12.243,9.276
math,31.1%,19.1%,-12.0 pp,0.311,0.191,156.594,130.187,3.887,3.641


### Interpretation

- If math performance drops sharply without Nemotron, then Nemotron was driving the pooled math advantage.
- If factor directions remain similar, then the finding is robust.
- If factor directions flip, then the previous conclusion should be written as model-dependent rather than general.